# 05 - PaddleOCR Plate Recognizer Fine-Tuning

Fine-tune `en_PP-OCRv5_mobile_rec` pada crop plat Test V2 yang sudah dilabel. Train dipakai untuk optimasi, valid untuk memilih checkpoint, dan test hanya untuk evaluasi final.

## 1. Environment and paths

Gunakan kernel `.venv`. Tool training berada pada source resmi PaddleOCR di `TEMPORARY/PaddleOCR` karena package wheel hanya menyertakan API inference.

In [ ]:
import json
import re
import subprocess
import sys
import urllib.request
from difflib import SequenceMatcher
from pathlib import Path
from time import perf_counter

import cv2
import numpy as np
import torch  # Load Torch DLLs before Paddle on Windows to avoid WinError 127.
import paddle
import pandas as pd
import yaml
from IPython.display import display
from paddleocr import TextRecognition

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ml').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Project root tidak ditemukan.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_ROOT = PROJECT_ROOT / 'data/processed/plate_ocr'
MANIFEST_ROOT = DATA_ROOT / 'manifests'
PADDLEOCR_ROOT = PROJECT_ROOT / 'TEMPORARY/PaddleOCR'
EXPERIMENT_ROOT = PROJECT_ROOT / 'ml/experiments/paddleocr_plate_recognizer'
PRETRAINED_ROOT = PROJECT_ROOT / 'TEMPORARY/pretrained_models'
CONFIG_PATH = EXPERIMENT_ROOT / 'plate_recognizer_en_v2.yml'
OUTPUT_ROOT = EXPERIMENT_ROOT / 'output_en_v2'
EXPORT_ROOT = EXPERIMENT_ROOT / 'inference_en_v2'
PRETRAINED_PATH = PRETRAINED_ROOT / 'en_PP-OCRv5_mobile_rec_pretrained.pdparams'
BASE_CONFIG_PATH = PADDLEOCR_ROOT / 'configs/rec/PP-OCRv5/multi_language/en_PP-OCRv5_mobile_rec.yaml'
PADDLEOCR_RUNNER = PROJECT_ROOT / 'ml/training/paddleocr_runner.py'
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
PRETRAINED_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Project: {PROJECT_ROOT}')
print(f'Paddle: {paddle.__version__} | CUDA build: {paddle.device.is_compiled_with_cuda()} | device: {paddle.device.get_device()}')


## 2. One-time setup

Laptop ini memiliki NVIDIA RTX 4060, tetapi environment awal memakai Paddle CPU. Untuk GPU, tutup semua kernel Jupyter, jalankan perintah berikut di terminal `.venv`, kemudian buka notebook dan restart kernel. Driver 566 memenuhi persyaratan CUDA 12.6 PaddlePaddle.

In [ ]:
# Jalankan di terminal, bukan saat kernel ini masih aktif:
# python -m pip uninstall -y paddlepaddle paddlepaddle-gpu
# python -m pip install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
# python -m pip install shapely scikit-image pyclipper lmdb rapidfuzz cython simsimd stringzilla
# python -m pip install albucore==0.0.24 albumentations==2.0.8 --no-deps
# python -c "import paddle; print(paddle.device.is_compiled_with_cuda(), paddle.device.get_device())"


In [ ]:
required_paths = [BASE_CONFIG_PATH, PADDLEOCR_ROOT / 'tools/train.py', MANIFEST_ROOT / 'train.txt', MANIFEST_ROOT / 'valid.txt', MANIFEST_ROOT / 'test.txt']
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(str(path) for path in missing_paths))

def read_manifest(path: Path) -> pd.DataFrame:
    rows = []
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
        if not line.strip():
            continue
        parts = line.split('\t', maxsplit=1)
        if len(parts) != 2:
            raise ValueError(f'Invalid manifest row {path.name}:{line_number}')
        relative_path, label = parts
        rows.append({'relative_path': relative_path, 'label': label, 'image_path': DATA_ROOT / relative_path})
    return pd.DataFrame(rows)

manifests = {split: read_manifest(MANIFEST_ROOT / f'{split}.txt') for split in ('train', 'valid', 'test')}
for split, frame in manifests.items():
    missing_images = [path for path in frame['image_path'] if not path.is_file()]
    invalid_labels = frame[~frame['label'].str.fullmatch(r'[A-Z]{1,3}[0-9]{1,4}[A-Z]{0,3}')]
    print(f'{split}: {len(frame)} labels | missing images={len(missing_images)} | invalid labels={len(invalid_labels)}')
    if missing_images or not invalid_labels.empty:
        raise ValueError(f'{split} manifest validation failed.')

train_labels = set(manifests['train']['label'])
test_overlap = train_labels.intersection(manifests['test']['label'])
print(f'Identical plate texts shared by train/test: {len(test_overlap)} (allowed; image leakage is checked separately)')


## 3. Download official pretrained training weights

In [ ]:
PRETRAINED_URL = 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/en_PP-OCRv5_mobile_rec_pretrained.pdparams'
if not PRETRAINED_PATH.is_file():
    print('Downloading official pretrained weights...')
    urllib.request.urlretrieve(PRETRAINED_URL, PRETRAINED_PATH)
print(f'Pretrained weights: {PRETRAINED_PATH} ({PRETRAINED_PATH.stat().st_size / 1_000_000:.1f} MB)')


## 4. Baseline on labeled test

Baseline memakai model resmi sebelum fine-tuning dan satu crop rectified per sampel. Test tidak digunakan untuk memilih preprocessing atau checkpoint.

In [ ]:
RUN_BASELINE = True
BASELINE_CSV = EXPERIMENT_ROOT / 'baseline_test_results.csv'

def normalize_text(value: str) -> str:
    return re.sub(r'[^A-Z0-9]', '', str(value).upper())

def similarity(prediction: str, ground_truth: str) -> float:
    return SequenceMatcher(None, prediction, ground_truth).ratio()

def paddle_result_payload(result) -> dict:
    payload = result.json
    return payload.get('res', payload)

if RUN_BASELINE and not BASELINE_CSV.is_file():
    baseline_model = TextRecognition(model_name='en_PP-OCRv5_mobile_rec', device='gpu:0' if paddle.device.is_compiled_with_cuda() else 'cpu')
    baseline_rows = []
    for index, row in enumerate(manifests['test'].itertuples(index=False), start=1):
        image = cv2.imread(str(row.image_path))
        started_at = perf_counter()
        outputs = list(baseline_model.predict(input=image, batch_size=1))
        latency_ms = (perf_counter() - started_at) * 1000
        payload = paddle_result_payload(outputs[0]) if outputs else {}
        prediction = normalize_text(payload.get('rec_text', ''))
        baseline_rows.append({'file': row.relative_path, 'ground_truth': row.label, 'prediction': prediction, 'confidence': float(payload.get('rec_score', 0.0)), 'exact_match': prediction == row.label, 'character_similarity': similarity(prediction, row.label), 'latency_ms': latency_ms})
        if index % 25 == 0:
            print(f'Baseline {index}/{len(manifests["test"])}')
    pd.DataFrame(baseline_rows).to_csv(BASELINE_CSV, index=False)

baseline_df = pd.read_csv(BASELINE_CSV) if BASELINE_CSV.is_file() else pd.DataFrame()
if not baseline_df.empty:
    print(f'Baseline exact accuracy: {baseline_df["exact_match"].mean():.2%}')
    print(f'Baseline character similarity: {baseline_df["character_similarity"].mean():.2%}')
    print(f'Baseline median latency: {baseline_df["latency_ms"].median():.1f} ms/image')


## 5. Generate fine-tuning configuration

In [ ]:
with BASE_CONFIG_PATH.open('r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

use_gpu = paddle.device.is_compiled_with_cuda()
config['Global'].update({
    'use_gpu': use_gpu,
    'epoch_num': 30,
    'print_batch_step': 10,
    'save_model_dir': str(OUTPUT_ROOT),
    'save_epoch_step': 5,
    'eval_batch_step': [0, 50],
    'pretrained_model': str(PRETRAINED_PATH),
    'checkpoints': None,
    'character_dict_path': str(PADDLEOCR_ROOT / 'ppocr/utils/dict/ppocrv5_en_dict.txt'),
    'max_text_length': 10,
    'use_space_char': True,
    'use_visualdl': False,
    'distributed': False,
})
config['Optimizer']['lr']['learning_rate'] = 0.0001
config['Optimizer']['lr']['warmup_epoch'] = 2
config['Architecture']['Head']['head_list'][1]['NRTRHead']['max_text_length'] = 10
for transform in config['Train']['dataset']['transforms']:
    if 'RecConAug' in transform:
        transform['RecConAug']['max_text_length'] = 10
config['Train']['dataset']['data_dir'] = str(DATA_ROOT)
config['Train']['dataset']['label_file_list'] = [str(MANIFEST_ROOT / 'train.txt')]
config['Train']['sampler']['first_bs'] = 32
config['Train']['loader'].update({'batch_size_per_card': 32, 'num_workers': 0, 'drop_last': True})
config['Eval']['dataset']['data_dir'] = str(DATA_ROOT)
config['Eval']['dataset']['label_file_list'] = [str(MANIFEST_ROOT / 'valid.txt')]
config['Eval']['loader'].update({'batch_size_per_card': 32, 'num_workers': 0, 'drop_last': False})

with CONFIG_PATH.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, sort_keys=False, allow_unicode=False)
print(f'Config written: {CONFIG_PATH}')
print(f'Training device: {"GPU" if use_gpu else "CPU"}')


## 6. Train

Training sengaja menolak CPU secara default. Ubah `ALLOW_CPU_TRAINING=True` hanya jika siap menunggu jauh lebih lama. Output training tampil langsung di cell dan checkpoint terbaik disimpan sebagai `best_accuracy`.

In [ ]:
RUN_TRAINING = True
ALLOW_CPU_TRAINING = False

if RUN_TRAINING:
    if not paddle.device.is_compiled_with_cuda() and not ALLOW_CPU_TRAINING:
        raise RuntimeError('PaddlePaddle masih CPU build. Jalankan instalasi GPU pada bagian setup, restart kernel, lalu ulangi dari cell pertama.')
    command = [sys.executable, str(PADDLEOCR_RUNNER), str(PADDLEOCR_ROOT / 'tools/train.py'), '-c', str(CONFIG_PATH)]
    print('Running:', ' '.join(command))
    subprocess.run(command, cwd=PADDLEOCR_ROOT, check=True)
else:
    print('Training skipped.')


## 7. Evaluate best checkpoint on valid, then test

In [ ]:
BEST_CHECKPOINT = OUTPUT_ROOT / 'best_accuracy'
if not BEST_CHECKPOINT.with_suffix('.pdparams').is_file():
    raise FileNotFoundError(f'Best checkpoint not found: {BEST_CHECKPOINT}.pdparams')

def run_official_eval(manifest_path: Path) -> None:
    command = [sys.executable, str(PADDLEOCR_RUNNER), str(PADDLEOCR_ROOT / 'tools/eval.py'), '-c', str(CONFIG_PATH), '-o', f'Global.checkpoints={BEST_CHECKPOINT}', f'Eval.dataset.label_file_list=[{manifest_path}]']
    print('Running:', ' '.join(command))
    subprocess.run(command, cwd=PADDLEOCR_ROOT, check=True)

print('VALID EVALUATION')
run_official_eval(MANIFEST_ROOT / 'valid.txt')
print('TEST EVALUATION - final holdout')
run_official_eval(MANIFEST_ROOT / 'test.txt')


## 8. Export inference model

In [ ]:
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
export_command = [sys.executable, str(PADDLEOCR_RUNNER), str(PADDLEOCR_ROOT / 'tools/export_model.py'), '-c', str(CONFIG_PATH), '-o', f'Global.pretrained_model={BEST_CHECKPOINT}', f'Global.save_inference_dir={EXPORT_ROOT}']
print('Running:', ' '.join(export_command))
subprocess.run(export_command, cwd=PADDLEOCR_ROOT, check=True)
print(f'Inference model exported: {EXPORT_ROOT}')


## Decision after experiment

Bandingkan exact accuracy checkpoint dengan baseline. Jika valid dan test meningkat tetapi belum cukup, label train tambahan sampai 2.000. Jangan mengubah test labels, preprocessing, atau checkpoint berdasarkan hasil test; seluruh keputusan iterasi menggunakan valid.